In [22]:
import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple, Callable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from symdisc import generate_euclidean_killing_fields_with_names
from symdisc.discovery import(
make_model_jacobian_callable_torch,
getEquivariantResidualMatrix,
discover_symmetry_coeffs
)

from symdisc.enforcement.regularization.utilities import as_field_lastdim
#from symdisc.enforcement.regularization.diagonal import sum_fields
#from symdisc.enforcement.regularization.penalties import forward_with_equivariance_penalty

In [2]:
# =============================================================================
# Data
# =============================================================================

class GradUDataset(Dataset):
    """A minimal dataset of gradU vectors.

    Provide either:
      - x: torch.Tensor of shape (N,9)
    or
      - load from a CSV externally and pass the tensor here.

    """
    def __init__(self, x: torch.Tensor):
        assert x.ndim == 2 and x.shape[1] == 9
        self.x = x

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx]

In [3]:
# =============================================================================
# Pope basis: gradU(9) -> (10,9)
# =============================================================================

def _decompose_grad_u(x9: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U = x9.view(*x9.shape[:-1], 3, 3)
    S = 0.5 * (U + U.transpose(-1, -2))
    W = 0.5 * (U - U.transpose(-1, -2))
    return S, W


def pope_basis_10x9(x9: torch.Tensor, *, normalize_SW: bool = True, eps: float = 1e-8) -> torch.Tensor:
    """x9: (N,9) -> (N,10,9) Pope basis tensors flattened row-major."""
    S, W = _decompose_grad_u(x9)

    if normalize_SW:
        nS = torch.sqrt((S * S).sum(dim=(-1, -2), keepdim=True)).clamp_min(eps)
        nW = torch.sqrt((W * W).sum(dim=(-1, -2), keepdim=True)).clamp_min(eps)
        S = S / nS
        W = W / nW

    I = torch.eye(3, device=x9.device, dtype=x9.dtype).view(1, 3, 3)

    S2 = S @ S
    W2 = W @ W

    trS2 = torch.einsum('bii->b', S2)
    trW2 = torch.einsum('bii->b', W2)

    T1  = S
    T2  = S @ W - W @ S
    T3  = S2 - (trS2.view(-1, 1, 1) / 3.0) * I
    T4  = W2 - (trW2.view(-1, 1, 1) / 3.0) * I
    T5  = W @ S2 - S2 @ W
    T6  = W2 @ S + S @ W2 - (2.0 / 3.0) * torch.einsum('bii->b', S @ W2).view(-1, 1, 1) * I
    T7  = W @ S @ W2 - W2 @ S @ W
    T8  = S @ W @ S2 - S2 @ W @ S
    T9  = W2 @ S2 + S2 @ W2 - (2.0 / 3.0) * torch.einsum('bii->b', S2 @ W2).view(-1, 1, 1) * I
    T10 = W @ S2 @ W2 - W2 @ S2 @ W

    basis = torch.stack([T1,T2,T3,T4,T5,T6,T7,T8,T9,T10], dim=1)
    basis = 0.5 * (basis + basis.transpose(-1, -2))

    return basis.reshape(basis.shape[0], 10, 9)

In [6]:
# =============================================================================
# Vector fields: Euclidean Killing fields on R^9
# =============================================================================

def get_euclidean_killing_fields_d9(
    *,
    include_translations: bool = True,
    include_rotations: bool = True,
) -> Tuple[List[Callable], List[str]]:
    """Return list of vector fields vf(x)->(…,9) and their names."""
    fields, names = generate_euclidean_killing_fields_with_names(
        d=9,
        include_translations=include_translations,
        include_rotations=include_rotations,
        backend='torch'
    )
    fields = [as_field_lastdim(f, d=9) for f in fields]
    return fields, names


def lift_out_field_9_to_90(field9: Callable) -> Callable:
    """Lift a 9D field to act block-diagonally on a flattened (10×9)=90D output.

    Input y_flat: (...,90)
    Reshape to (...,10,9), apply field9 to last dim, reshape back.
    """
    def fld90(y_flat: torch.Tensor, *, meta=None) -> torch.Tensor:
        y = y_flat.view(*y_flat.shape[:-1], 10, 9)
        v = field9(y, meta=meta)  # (...,10,9)
        return v.reshape(*y_flat.shape[:-1], 90)
    return fld90


def linear_combo_field(fields: Sequence[Callable], coeff: torch.Tensor) -> Callable:
    assert coeff.ndim == 1 and coeff.numel() == len(fields)
    def Fld(x: torch.Tensor, *, meta=None) -> torch.Tensor:
        out = 0.0
        for a, f in zip(coeff, fields):
            out = out + a * f(x, meta=meta)
        return out
    return Fld

In [16]:
# =============================================================================
# Jacobian callable: J_F(X) -> (N, p, d) with p=90 and d=9
# =============================================================================

def make_model_jacobian_callable_torch(
    model: Callable[[torch.Tensor], torch.Tensor],
    *,
    create_graph: bool = False,
) -> Callable[[torch.Tensor], torch.Tensor]:
    """Jacobian of model: x->(N,10,9) flattened to (N,90)."""

    def J(X: torch.Tensor) -> torch.Tensor:
        X = X.requires_grad_(True)
        Y = model(X)
        Yf = Y.reshape(Y.shape[0], -1)  # (N,90)
        N, p = Yf.shape
        d = X.shape[1]
        J_out = torch.zeros(N, p, d, device=X.device, dtype=X.dtype)
        # Compute per-output gradients
        for j in range(p):
            grad = torch.autograd.grad(
                outputs=Yf[:, j].sum(),
                inputs=X,
                retain_graph=True,
                create_graph=create_graph,
            )[0]
            J_out[:, j, :] = grad
        return J_out.detach() if not create_graph else J_out

    return J
    

class PopeOracle(nn.Module):
    def __init__(self, normalize_SW=True):
        super().__init__()
        self.normalize_SW = normalize_SW

    def forward(self, x):  # x: (N,9)
        # returns (N,10,9)
        return pope_basis_10x9(x, normalize_SW=self.normalize_SW)

class PopeOracleFlat(nn.Module):
    def __init__(self, normalize_SW=True):
        super().__init__()
        self.normalize_SW = normalize_SW

    def forward(self, x):  # x: (N,9)
        # returns (N,90)
        y = pope_basis_10x9(x, normalize_SW=self.normalize_SW)
        return y.reshape(x.shape[0], 90)

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#
# X_eval: (N,9)
X_eval = torch.rand((1000,9))
                    
# Oracle map for discovery
def F_oracle(x):
    return pope_basis_10x9(x, normalize_SW=True)
    
# Flatten oracle to (N,90)
F_flat = lambda x: F_oracle(x).reshape(x.shape[0], 90)
#J = make_model_jacobian_callable_torch(F_oracle, create_graph=False)

# Build vf bases
vf9, names = get_euclidean_killing_fields_d9(include_translations=False, include_rotations=True)
vf_out90 = [lift_out_field_9_to_90(v) for v in vf9]

# Jacobian must align with flattened output (N,90,9)
#def J_flat(X):
#    # J of (N,10,9) flattened => (N,90,9)
#    return J(X)  # J already built from F_oracle returns (N,90,9)



oracle_flat = PopeOracleFlat(normalize_SW=True).to(device)

J = make_model_jacobian_callable_torch(
    oracle_flat,              # <-- nn.Module now
    #batch_size=256,
    create_graph=False
)

def J_flat(X):
    return J(X)               # (N,90,9)


M, info = getEquivariantResidualMatrix(
    X=X_eval.to(device),
    F=F_flat,
    J_F=J_flat,
    vf_in=vf9,
    vf_out=vf_out90,
    coupling='aligned',
    normalize_rows=True,
)

In [23]:
C, svals = discover_symmetry_coeffs(M, backend="torch")
#print(C)
print(svals)

#print("Discovered coefficient matrix C (q × r):", tuple(C.shape))
#print("Small singular values:", svals if isinstance(svals, np.ndarray) else svals.detach().cpu().numpy())
#print("Field names:", names)

# Pretty print top contributors
def print_top_components(C, names, topk=4):
    C_np = C.detach().cpu().numpy() if hasattr(C, "detach") else np.asarray(C)
    for j in range(C_np.shape[1]):
        coeffs = C_np[:, j]
        order = np.argsort(-np.abs(coeffs))
        print(f"\nSymmetry #{j+1}:")
        for idx in order[:topk]:
            print(f"  {names[idx]:>6s}: {coeffs[idx]: .4f}")

print_top_components(C, names, topk=6)

tensor([7.6875e-06, 8.9746e-06, 3.2573e-05, 2.5861e+01, 2.5977e+01, 2.6339e+01,
        2.8755e+01, 2.8799e+01, 2.9849e+01, 3.0217e+01, 3.0598e+01, 3.1158e+01,
        3.1554e+01, 3.1913e+01, 3.2325e+01, 3.2563e+01, 3.5609e+01, 3.6832e+01,
        3.7270e+01, 3.7506e+01, 3.7938e+01, 3.9207e+01, 3.9550e+01, 3.9997e+01,
        4.0437e+01, 4.1557e+01, 4.2421e+01, 4.2535e+01, 4.9717e+01, 4.9996e+01,
        5.9721e+01, 6.0078e+01, 6.1599e+01, 1.1707e+02, 1.2200e+02, 1.2415e+02],
       grad_fn=<IndexSelectBackward0>)

Symmetry #1:
   R_0_2: -0.3408
   R_3_5: -0.3408
   R_2_8: -0.3408
   R_0_6: -0.3408
   R_1_7: -0.3408
   R_6_8: -0.3408

Symmetry #2:
   R_3_6:  0.3408
   R_4_5:  0.3408
   R_1_2:  0.3408
   R_5_8:  0.3408
   R_7_8:  0.3408
   R_4_7:  0.3408

Symmetry #3:
   R_3_4:  0.4082
   R_0_3:  0.4082
   R_2_5:  0.4082
   R_6_7:  0.4082
   R_0_1:  0.4082
   R_1_4:  0.4082

Symmetry #4:
   R_0_4: -0.4592
   R_4_8: -0.4147
   R_3_7: -0.3468
   R_1_5: -0.2733
   R_2_7:  0.2544
   R_2_3: 

/home/ben/Documents/GitHub/SymmetryML/src/symdisc/discovery/core.py:99: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  if float(Ssum) <= 0.0:
